In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input", topdown=True):
    for name in files:
        print(os.path.join(root, name))


/kaggle/input/fraud-detection/fraudTest.csv
/kaggle/input/fraud-detection/fraudTrain.csv


In [2]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/fraud-detection/fraudTrain.csv")
test_df = pd.read_csv("/kaggle/input/fraud-detection/fraudTest.csv")

train_df.head(), test_df.head()


(   Unnamed: 0 trans_date_trans_time            cc_num  \
 0           0   2019-01-01 00:00:18  2703186189652095   
 1           1   2019-01-01 00:00:44      630423337322   
 2           2   2019-01-01 00:00:51    38859492057661   
 3           3   2019-01-01 00:01:16  3534093764340240   
 4           4   2019-01-01 00:03:06   375534208663984   
 
                              merchant       category     amt      first  \
 0          fraud_Rippin, Kub and Mann       misc_net    4.97   Jennifer   
 1     fraud_Heller, Gutmann and Zieme    grocery_pos  107.23  Stephanie   
 2                fraud_Lind-Buckridge  entertainment  220.11     Edward   
 3  fraud_Kutch, Hermiston and Farrell  gas_transport   45.00     Jeremy   
 4                 fraud_Keeling-Crist       misc_pos   41.96      Tyler   
 
       last gender                        street  ...      lat      long  \
 0    Banks      F                561 Perry Cove  ...  36.0788  -81.1781   
 1     Gill      F  43039 Riley Greens S

In [3]:
# Copy so we don't touch original accidentally
train = train_df.copy()
test = test_df.copy()

# Target
y_train = train["is_fraud"]
y_test = test["is_fraud"]

# Features (remove target column)
X_train = train.drop("is_fraud", axis=1)
X_test = test.drop("is_fraud", axis=1)

X_train.shape, X_test.shape


((1296675, 22), (555719, 22))

In [6]:
possible_cols_to_drop = [
    "Unnamed: 0",
    "trans_date_trans_time",
    "cc_num",
    "first",
    "last",
    "street",
    "job",
    "dob",
    "trans_num"
]

for col in possible_cols_to_drop:
    if col in X_train.columns:
        X_train = X_train.drop(col, axis=1)
    if col in X_test.columns:
        X_test = X_test.drop(col, axis=1)

X_train.shape, X_test.shape


((1296675, 13), (555719, 13))

In [7]:
combined = pd.concat([X_train, X_test], axis=0)

combined_encoded = pd.get_dummies(
    combined,
    columns=["category", "gender"],
    drop_first=True  # avoid dummy trap
)

# split back
X_train_encoded = combined_encoded.iloc[:len(X_train), :].copy()
X_test_encoded = combined_encoded.iloc[len(X_train):, :].copy()

X_train_encoded.shape, X_test_encoded.shape


((1296675, 25), (555719, 25))

In [9]:
bad_cols = ["merchant", "city", "state", "zip"]

for col in bad_cols:
    if col in X_train_encoded.columns:
        X_train_encoded = X_train_encoded.drop(col, axis=1)
    if col in X_test_encoded.columns:
        X_test_encoded = X_test_encoded.drop(col, axis=1)

X_train_encoded.dtypes.unique()


array([dtype('float64'), dtype('int64'), dtype('bool')], dtype=object)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train_encoded, y_train)


LogisticRegression(class_weight='balanced', max_iter=500, n_jobs=-1)

In [11]:
y_pred = model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.8602801055929346

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.86      0.92    553574
           1       0.00      0.10      0.01      2145

    accuracy                           0.86    555719
   macro avg       0.50      0.48      0.47    555719
weighted avg       0.99      0.86      0.92    555719



In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_encoded, y_train)

rf_pred = rf.predict(X_test_encoded)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("\nRF Classification Report:\n")
print(classification_report(y_test, rf_pred))


RF Accuracy: 0.9973025935769697

RF Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.92      0.33      0.49      2145

    accuracy                           1.00    555719
   macro avg       0.96      0.67      0.74    555719
weighted avg       1.00      1.00      1.00    555719

